# SENTINEL ROVER / Colab実行ノートブック

AIが設備の異常を判別し、危険区域を避けて巡回する点検ロボットです。
GPU・APIキーは不要です。ランタイムはCPUのままで構いません。
このノートブックのColab UI上での実行は未検証です。同じPythonコードはLinuxで実行・検証しています。

1. 同梱のZIPをアップロードします。
2. 依存関係を入れ、物理シミュレーション・分類・録画を実行します。
3. テストし、PDFを再生成し、結果をダウンロードします。

ノイズを加えた評価は合成データです。実機性能を示すものではありません。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # SentinelRover_final_package.zip を選択
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('一式ZIPを選択してください')
work = Path('/content/sentinel_work').resolve()
work.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    for member in archive.infolist():
        target = (work / member.filename).resolve()
        if not target.is_relative_to(work):
            raise ValueError('安全でないZIP内パスです')
    archive.extractall(work)
project = work / 'SentinelRover'
if not (project / 'main.py').is_file():
    raise FileNotFoundError('ZIP内にSentinelRover/main.pyが見つかりません')
print(project)


In [ ]:
import os, subprocess, sys
os.environ['CC'] = 'gcc'
os.environ['CXX'] = 'g++'
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'build-essential'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project/'requirements.txt')], check=True)


## 学習・評価・動画生成

地図と人の軌道は同じまま、センサーseed 100〜104を比較します。
依存関係のビルドやCPUの混雑により時間がかかる場合があります。
動画を省略して先に試す場合は `--video` を削除してください。


In [ ]:
subprocess.run([sys.executable, 'main.py', '--evaluate', '--video'], cwd=project, check=True)
subprocess.run([sys.executable, 'tests.py'], cwd=project, check=True)
subprocess.run([sys.executable, 'report.py'], cwd=project, check=True)


In [ ]:
import json
from IPython.display import Video, display
metrics = json.loads((project/'results/ml_metrics.json').read_text())
print('通常の合成テスト正解率:', metrics['test']['accuracy'])
print('ストレス条件の正解率:', metrics['stress']['accuracy'])
display(Video(str(project/'results/SentinelRover_demo.mp4'), embed=True))


## 指定ファイル名の準備とダウンロード

個人名やメールではなくOmniCampusアカウントIDを入力します。
GitHub・YouTubeへの公開やomnicampusへの提出は、このノートブックでは行いません。
SUBMISSION_GUIDE.mdを確認してください。


In [ ]:
account_id = 'YOUR_OMNICAMPUS_ID'  # 自分の講座アカウントIDへ変更
subprocess.run([sys.executable, 'prepare_submission.py', '--account-id', account_id], cwd=project, check=True)
pdf = project/'submission_ready'/f'説明資料_PAI最終課題_{account_id}.pdf'
files.download(str(pdf))
files.download(str(project/'results/SentinelRover_demo.mp4'))
